In [1]:
# Cell 1: Load data and analysis results
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "transactions_enriched.csv")
df['date_dt'] = pd.to_datetime(df['date_iso'])

print(f"Loaded: {df.shape[0]} transactions")

Loaded: 193 transactions


In [2]:
# Cell 2: Template-Based Narrative Engine
# WHY: This generates plain-English spending insights without any
# API calls. It uses the same data patterns we detected in Phase 2
# and translates them into sentences a human would write.
#
# This is how Monzo, Revolut, and most fintech apps actually work.
# An LLM is overkill for structured financial summaries.

def generate_narrative(df):
    """
    Takes a transaction DataFrame and returns a complete
    plain-English financial narrative as a string.
    """
    spending = df[df['direction'] == 'OUT'].copy()
    income = df[df['direction'] == 'IN'].copy()
    spending['month'] = pd.to_datetime(spending['date_iso']).dt.to_period('M')
    
    # Core numbers
    total_in = income['money_in'].sum()
    total_out = spending['money_out'].sum()
    months = spending['month'].nunique()
    avg_monthly_in = total_in / max(months, 1)
    avg_monthly_out = total_out / max(months, 1)
    net = avg_monthly_in - avg_monthly_out
    
    # Top categories
    cat_monthly = spending.groupby('category')['money_out'].sum() / max(months, 1)
    cat_monthly = cat_monthly.sort_values(ascending=False)
    
    # Top merchants
    merch_monthly = spending.groupby('merchant')['money_out'].sum() / max(months, 1)
    merch_monthly = merch_monthly.sort_values(ascending=False)
    
    # Subscriptions detection
    merchant_months = spending.groupby('merchant')['month'].nunique()
    merchant_avg = spending.groupby('merchant')['money_out'].mean()
    likely_subs = merchant_months[merchant_months >= max(months - 1, 2)].index
    sub_cost = sum(merchant_avg[m] for m in likely_subs if merchant_avg[m] < 50)
    
    # Month-over-month (if we have 2+ months)
    months_list = sorted(spending['month'].unique())
    mom_insights = []
    if len(months_list) >= 2:
        m1 = spending[spending['month'] == months_list[-2]]
        m2 = spending[spending['month'] == months_list[-1]]
        m1_by_cat = m1.groupby('category')['money_out'].sum()
        m2_by_cat = m2.groupby('category')['money_out'].sum()
        
        for cat in set(list(m1_by_cat.index) + list(m2_by_cat.index)):
            prev = m1_by_cat.get(cat, 0)
            curr = m2_by_cat.get(cat, 0)
            if prev > 0 and curr > 0:
                change_pct = (curr - prev) / prev * 100
                if abs(change_pct) > 30:
                    direction = "increased" if change_pct > 0 else "decreased"
                    mom_insights.append((cat, direction, abs(change_pct), prev, curr))
            elif prev == 0 and curr > 10:
                mom_insights.append((cat, "appeared for the first time", 0, 0, curr))
    
    # Anomaly detection
    anomalies = []
    for cat in spending['category'].unique():
        cat_txns = spending[spending['category'] == cat]
        if len(cat_txns) >= 3:
            mean = cat_txns['money_out'].mean()
            std = cat_txns['money_out'].std()
            threshold = mean + 2 * std
            for _, row in cat_txns.iterrows():
                if row['money_out'] > threshold and row['money_out'] > 20:
                    anomalies.append((row['date_iso'], row['merchant'], row['money_out'], mean, cat))
    anomalies.sort(key=lambda x: -x[2])
    
    # === BUILD THE NARRATIVE ===
    sections = []
    
    # Opening summary
    if net >= 0:
        opening = (
            f"Over the past {months} months, you earned an average of "
            f"GBP {avg_monthly_in:,.2f} per month and spent GBP {avg_monthly_out:,.2f}, "
            f"leaving a monthly surplus of GBP {net:,.2f}. "
            f"That is a healthy position -- you are spending within your means."
        )
    else:
        opening = (
            f"Over the past {months} months, you earned an average of "
            f"GBP {avg_monthly_in:,.2f} per month but spent GBP {avg_monthly_out:,.2f}, "
            f"which means you are spending GBP {abs(net):,.2f} more than you earn each month. "
            f"This is not sustainable long-term and is worth addressing."
        )
    sections.append(("Overview", opening))
    
    # Where the money goes
    top3 = list(cat_monthly.head(3).items())
    lifestyle_cats = ['Transport', 'Groceries', 'Eating Out', 'Shopping', 
                      'Food Delivery', 'Coffee & Cafe']
    lifestyle_spend = sum(cat_monthly.get(c, 0) for c in lifestyle_cats)
    
    where_text = (
        f"Your biggest spending categories are {top3[0][0]} "
        f"(GBP {top3[0][1]:,.2f}/month), {top3[1][0]} "
        f"(GBP {top3[1][1]:,.2f}/month), and {top3[2][0]} "
        f"(GBP {top3[2][1]:,.2f}/month). "
        f"Your total lifestyle spending -- covering transport, food, shopping, "
        f"and daily expenses -- averages GBP {lifestyle_spend:,.2f} per month."
    )
    
    # Add top merchant insight
    top_merchant = merch_monthly.index[0]
    top_merchant_amt = merch_monthly.iloc[0]
    non_rent = merch_monthly[~merch_monthly.index.isin(['Anthem Homes (Rent)', '[REDACTED]'])]
    if len(non_rent) > 0:
        top_lifestyle_merchant = non_rent.index[0]
        top_lifestyle_amt = non_rent.iloc[0]
        where_text += (
            f" Your single biggest lifestyle merchant is {top_lifestyle_merchant}, "
            f"where you spend GBP {top_lifestyle_amt:,.2f} per month on average."
        )
    
    sections.append(("Where Your Money Goes", where_text))
    
    # Trends
    if mom_insights:
        trend_parts = []
        for cat, direction, pct, prev, curr in sorted(mom_insights, key=lambda x: -abs(x[4] - x[3])):
            if direction == "appeared for the first time":
                trend_parts.append(
                    f"{cat} spending appeared for the first time last month "
                    f"at GBP {curr:,.2f}"
                )
            else:
                trend_parts.append(
                    f"{cat} {direction} by {pct:.0f}% "
                    f"(from GBP {prev:,.2f} to GBP {curr:,.2f})"
                )
        
        trend_text = "Comparing your two most recent months: " + "; ".join(trend_parts[:4]) + "."
        sections.append(("Trends", trend_text))
    
    # Unusual transactions
    if anomalies:
        anom_text = f"We flagged {len(anomalies)} transactions as unusually large. "
        top_anom = anomalies[0]
        anom_text += (
            f"The biggest was GBP {top_anom[2]:,.2f} at {top_anom[1]} on {top_anom[0]}, "
            f"which is significantly higher than your average {top_anom[4]} spend of "
            f"GBP {top_anom[3]:,.2f}."
        )
        if len(anomalies) > 1:
            anom_text += f" Other flagged transactions include "
            others = [f"GBP {a[2]:,.2f} at {a[1]}" for a in anomalies[1:3]]
            anom_text += " and ".join(others) + "."
        sections.append(("Unusual Activity", anom_text))
    
    # Savings
    savings_suggestions = {
        'Transport': ('walking or cycling for short trips', 0.20),
        'Eating Out': ('meal prepping twice a week', 0.30),
        'Food Delivery': ('cooking instead of ordering', 0.50),
        'Coffee & Cafe': ('making coffee at home', 0.60),
        'Shopping': ('waiting 48 hours before non-essential purchases', 0.25),
    }
    
    total_annual_saving = 0
    save_parts = []
    for cat, (suggestion, pct) in savings_suggestions.items():
        monthly = cat_monthly.get(cat, 0)
        if monthly > 5:
            saving = monthly * pct * 12
            total_annual_saving += saving
            save_parts.append(f"GBP {saving:,.0f} on {cat.lower()} by {suggestion}")
    
    if save_parts and total_annual_saving > 100:
        save_text = (
            f"Based on your spending patterns, there are realistic opportunities "
            f"to save approximately GBP {total_annual_saving:,.0f} per year. "
            f"The biggest areas: " + "; ".join(save_parts[:3]) + "."
        )
        sections.append(("Savings Opportunities", save_text))
    
    # Build final output
    narrative = ""
    for title, text in sections:
        narrative += f"\n{title}\n{'─' * len(title)}\n{text}\n"
    
    return narrative


# Generate and display
narrative = generate_narrative(df)
print(narrative)


Overview
────────
Over the past 3 months, you earned an average of GBP 1,762.86 per month but spent GBP 1,774.89, which means you are spending GBP 12.04 more than you earn each month. This is not sustainable long-term and is worth addressing.

Where Your Money Goes
─────────────────────
Your biggest spending categories are Transfers (GBP 713.33/month), Rent (GBP 480.00/month), and Professional (GBP 173.99/month). Your total lifestyle spending -- covering transport, food, shopping, and daily expenses -- averages GBP 310.97 per month. Your single biggest lifestyle merchant is Remitly (Intl Transfer), where you spend GBP 221.67 per month on average.

Trends
──────
Comparing your two most recent months: Transport decreased by 91% (from GBP 162.90 to GBP 15.13); Shopping decreased by 73% (from GBP 59.35 to GBP 16.00); Food Delivery decreased by 75% (from GBP 38.20 to GBP 9.46); Bank Fees decreased by 35% (from GBP 2.86 to GBP 1.85).

Unusual Activity
────────────────
We flagged 8 transacti

In [3]:
# Data Quality Audit
# WHY: NaN and blank values cause crashes in JavaScript, broken
# charts, and wrong calculations. We fix them here in Python
# before the data ever reaches the frontend.

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "transactions_enriched.csv")

print("=== DATA QUALITY AUDIT ===\n")

# Check every column for NaN, blank, and empty values
print("NaN counts per column:")
print(df.isna().sum().to_string())

print("\n\nBlank string counts per column:")
for col in df.columns:
    blanks = (df[col] == '').sum()
    if blanks > 0:
        print(f"  {col}: {blanks}")

print("\n\nEmpty string after strip:")
for col in df.columns:
    if df[col].dtype == 'object':
        empty = df[col].apply(lambda x: str(x).strip() == '').sum()
        if empty > 0:
            print(f"  {col}: {empty}")

print("\n\nSample rows with NaN in money_in:")
print(df[df['money_in'].isna()].head(3)[['date_iso', 'merchant', 'money_in', 'money_out', 'balance']].to_string())

print("\n\nSample rows with NaN in money_out:")
print(df[df['money_out'].isna()].head(3)[['date_iso', 'merchant', 'money_in', 'money_out', 'balance']].to_string())

print("\n\nData types:")
print(df.dtypes.to_string())

print("\n\nUnique values in 'direction':", df['direction'].unique())
print("Unique values in 'type':", df['type'].unique())

=== DATA QUALITY AUDIT ===

NaN counts per column:
date             0
description      0
type             0
money_in       176
money_out       17
balance          0
date_iso         0
direction        0
merchant         0
category         0
month            0
date_dt          0
week             0
year_week        0
day_of_week      0


Blank string counts per column:


Empty string after strip:


Sample rows with NaN in money_in:
     date_iso   merchant  money_in  money_out  balance
0  2026-01-02       Uber       NaN       8.30    31.66
3  2026-01-02   Chopstix       NaN       4.22    87.44
4  2026-01-02  NIS Store       NaN       3.29    84.15


Sample rows with NaN in money_out:
      date_iso    merchant  money_in  money_out  balance
1   2026-01-02  [REDACTED]      10.0        NaN    41.66
2   2026-01-02  [REDACTED]      50.0        NaN    91.66
16  2026-01-05  [REDACTED]       6.8        NaN     6.38


Data types:
date               str
description        str
type               st

In [4]:
# Fix: Replace NaN with 0 in money columns
# WHY: JavaScript cannot do math with null/NaN values.
# A spending transaction with money_in = 0 is correct and safe.
# A spending transaction with money_in = NaN will crash the frontend.

df['money_in'] = df['money_in'].fillna(0)
df['money_out'] = df['money_out'].fillna(0)

# Also drop the extra columns that were added during analysis
# The frontend only needs the core columns
frontend_columns = ['date_iso', 'description', 'merchant', 'category', 
                    'type', 'money_in', 'money_out', 'balance', 'direction']
df_clean = df[frontend_columns].copy()

# Verify
print("=== AFTER CLEANUP ===\n")
print("NaN counts:")
print(df_clean.isna().sum().to_string())

print(f"\nShape: {df_clean.shape}")
print(f"\nSample (first 5):")
print(df_clean.head().to_string())

# Save the frontend-ready version
output_path = PROJECT_ROOT / "data" / "processed" / "transactions_frontend.csv"
df_clean.to_csv(output_path, index=False)

# Also save as JSON (React reads JSON natively)
json_path = PROJECT_ROOT / "data" / "processed" / "transactions_frontend.json"
df_clean.to_json(json_path, orient='records', indent=2)

print(f"\nSaved CSV: {output_path}")
print(f"Saved JSON: {json_path}")
print(f"\nZero NaN values. Frontend-safe data ready.")

=== AFTER CLEANUP ===

NaN counts:
date_iso       0
description    0
merchant       0
category       0
type           0
money_in       0
money_out      0
balance        0
direction      0

Shape: (193, 9)

Sample (first 5):
     date_iso         description    merchant    category type  money_in  money_out  balance direction
0  2026-01-02       UBER UK RIDES        Uber   Transport  DEB       0.0       8.30    31.66       OUT
1  2026-01-02          [REDACTED]  [REDACTED]      Income  FPI      10.0       0.00    41.66        IN
2  2026-01-02          [REDACTED]  [REDACTED]      Income  FPI      50.0       0.00    91.66        IN
3  2026-01-02  CHOPSTIX SOUTHAMPT    Chopstix  Eating Out  DEB       0.0       4.22    87.44       OUT
4  2026-01-02         N I S STORE   NIS Store   Groceries  DEB       0.0       3.29    84.15       OUT

Saved CSV: C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\processed\transactions_frontend.csv
Saved JSON: C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\pr